# Ablation 3 — Xception + FFT
Train: FF++ C23 | Test: FF++ C23 + Celeb-DF v2

**Architecture:** Spatial stream + FFT frequency stream (concatenation fusion). No SRM, no cross-attention.

---
## Cell 1 — Imports & Constants

In [1]:
print("Jobayer")

Jobayer


In [2]:
print("Ablation — Pre-built Dataset Edition")

import os, cv2, json, random
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import Xception
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau,
    ModelCheckpoint, LearningRateScheduler
)
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_auc_score, roc_curve, average_precision_score,
    precision_recall_curve
)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
from tqdm import tqdm

# ── Reproducibility ────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

# ── Constants ──────────────────────────────────────────────────────────────
IMG_SIZE    = 299
BATCH_SIZE  = 32
EPOCHS_P1   = 15
EPOCHS_P2   = 30
AUTOTUNE    = tf.data.AUTOTUNE
NUM_GRADCAM = 5

# ── Pre-built dataset root ─────────────────────────────────────────────────
DATASET_ROOT = "/kaggle/input/datasets/jfaisal/deepfake-dataset"  # <-- update if needed

TRAIN_DIR  = os.path.join(DATASET_ROOT, "ff_train")
VAL_DIR    = os.path.join(DATASET_ROOT, "ff_val")
FFTEST_DIR = os.path.join(DATASET_ROOT, "ff_test")
CELEB_DIR  = os.path.join(DATASET_ROOT, "celeb_test")
MANIFEST   = os.path.join(DATASET_ROOT, "dataset_manifest.json")

# ── Output directories ─────────────────────────────────────────────────────
CKPT_DIR = "/kaggle/working/ckpts"
FIG_DIR  = "/kaggle/working/paper_figures"
HIST_DIR = "/kaggle/working/history"

for d in [CKPT_DIR, FIG_DIR, HIST_DIR]:
    os.makedirs(d, exist_ok=True)

# ── GPU ────────────────────────────────────────────────────────────────────
gpus = tf.config.list_physical_devices('GPU')
for g in gpus:
    try: tf.config.experimental.set_memory_growth(g, True)
    except RuntimeError: pass

print(f"TF {tf.__version__} | GPUs: {len(gpus)}")
print(f"IMG_SIZE={IMG_SIZE} | BATCH={BATCH_SIZE} | P1={EPOCHS_P1} | P2={EPOCHS_P2}")
print("✓ Setup complete")


Ablation — Pre-built Dataset Edition


2026-04-21 12:05:32.854804: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776773133.079005      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776773133.149803      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776773133.607572      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776773133.607612      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776773133.607617      23 computation_placer.cc:177] computation placer alr

TF 2.19.0 | GPUs: 2
IMG_SIZE=299 | BATCH=32 | P1=15 | P2=30
✓ Setup complete


---
## Cell 2 — Verify Pre-built Dataset

No frame extraction needed — frames already on disk. This cell verifies paths and prints manifest stats.

In [3]:
def count_partition(directory):
    counts = {}
    for cls in ['fake', 'real']:
        cls_dir = os.path.join(directory, cls)
        if os.path.isdir(cls_dir):
            counts[cls] = len([
                f for f in os.listdir(cls_dir)
                if f.lower().endswith(('.jpg', '.jpeg', '.png'))
            ])
        else:
            counts[cls] = 0
    return counts

print("Dataset Manifest:")
if os.path.exists(MANIFEST):
    with open(MANIFEST) as f:
        manifest = json.load(f)
    print(f"  Seed         : {manifest.get('seed')}")
    print(f"  Image size   : {manifest.get('img_size')}px")
    print(f"  Frames/video : {manifest.get('frames_per_video')}")
    print(f"  JPEG quality : {manifest.get('jpeg_quality')}")
    print(f"  Split ratios : {manifest.get('split_ratios')}")
    print(f"  Class order  : {manifest.get('class_order')}")
    print(f"  Total frames : {manifest.get('total_frames'):,}")
    print(f"  Leakage note : {manifest.get('note')}")
else:
    print("  WARNING: manifest not found — proceeding with directory counts only")

print("\nFrame counts per partition:")
partitions = {
    'ff_train'  : TRAIN_DIR,
    'ff_val'    : VAL_DIR,
    'ff_test'   : FFTEST_DIR,
    'celeb_test': CELEB_DIR,
}
total = 0
for name, path in partitions.items():
    if not os.path.isdir(path):
        print(f"  {name:<12}: *** DIRECTORY NOT FOUND: {path}")
        continue
    c = count_partition(path)
    subtotal = sum(c.values())
    total += subtotal
    print(f"  {name:<12}: fake={c['fake']:,}  real={c['real']:,}  total={subtotal:,}")
print(f"  {'TOTAL':<12}: {total:,} frames")

train_c = count_partition(TRAIN_DIR)
ratio   = train_c['fake'] / max(train_c['real'], 1)
print(f"\nTraining class ratio (fake/real): {ratio:.3f}")
if abs(ratio - 1.0) > 0.15:
    print("  \u26a0\ufe0f  Imbalance detected — class_weight applied automatically")
else:
    print("  \u2713 Classes balanced")
print("\n\u2713 Dataset verified — ready to train")


Dataset Manifest:
  Seed         : 42
  Image size   : 299px
  Frames/video : 10
  JPEG quality : 95
  Split ratios : [0.7, 0.15, 0.15]
  Class order  : fake=0, real=1 (tf.keras alphabetical order)
  Total frames : 26,000
  Leakage note : Video-level split: all frames from one source video appear in exactly one partition. No train/test leakage.

Frame counts per partition:
  ff_train    : fake=7,000  real=7,000  total=14,000
  ff_val      : fake=1,500  real=1,500  total=3,000
  ff_test     : fake=1,500  real=1,500  total=3,000
  celeb_test  : fake=3,000  real=3,000  total=6,000
  TOTAL       : 26,000 frames

Training class ratio (fake/real): 1.000
  ✓ Classes balanced

✓ Dataset verified — ready to train


---
## Cell 3 — Data Pipeline

Reads frames directly from pre-built partitions. fake=0, real=1 (alphabetical).

In [4]:
def make_dataset(directory, augment=False):
    """
    Load dataset from directory with fake/ and real/ subdirs.
    Returns (dataset, class_names). fake=0, real=1 (alphabetical).
    """
    raw = tf.keras.utils.image_dataset_from_directory(
        directory, seed=SEED,
        image_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE, label_mode="binary")

    class_names = raw.class_names

    if augment:
        aug = tf.keras.Sequential([
            layers.RandomFlip("horizontal"),
            layers.RandomRotation(0.10),
            layers.RandomZoom(0.10),
            layers.RandomBrightness(0.15),
            layers.RandomContrast(0.15),
        ], name="augmentation")
        raw = (raw.map(lambda x, y: (aug(x, training=True), y),
                       num_parallel_calls=AUTOTUNE)
                  .shuffle(2000, reshuffle_each_iteration=True))

    return raw.prefetch(AUTOTUNE), class_names


# ── Build all four datasets ────────────────────────────────────────────────
train_ds,  train_cls = make_dataset(TRAIN_DIR,  augment=True)
val_ds,    _         = make_dataset(VAL_DIR,    augment=False)
fftest_ds, _         = make_dataset(FFTEST_DIR, augment=False)
celeb_ds,  _         = make_dataset(CELEB_DIR,  augment=False)

print("Class names:", train_cls, "\u2192 fake=0, real=1")
assert train_cls == ['fake', 'real'], (
    f"Unexpected class order: {train_cls}. "
    "Ensure subdirectory names are exactly 'fake' and 'real'.")

lbl    = np.concatenate([y.numpy() for _, y in train_ds])
n_fake = int((lbl == 0).sum())
n_real = int((lbl == 1).sum())
print(f"Train: {n_fake:,} fake  {n_real:,} real")

total_train  = n_fake + n_real
class_weight = {
    0: total_train / (2 * n_fake),
    1: total_train / (2 * n_real),
}
print(f"Class weights: {class_weight}")
print("\u2713 Data pipeline ready")


Found 14000 files belonging to 2 classes.


I0000 00:00:1776773201.677177      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1776773201.683026      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Found 3000 files belonging to 2 classes.
Found 3000 files belonging to 2 classes.
Found 6000 files belonging to 2 classes.
Class names: ['fake', 'real'] → fake=0, real=1
Train: 7,000 fake  7,000 real
Class weights: {0: 1.0, 1: 1.0}
✓ Data pipeline ready


---
## Cell 4 — Loss & Training Helpers

In [5]:
class LabelSmoothedBCE(tf.keras.losses.Loss):
    def __init__(self, smoothing=0.05, **kwargs):
        super().__init__(**kwargs)
        self.smoothing = smoothing
    def call(self, y_true, y_pred):
        y_true = y_true * (1 - self.smoothing) + 0.5 * self.smoothing
        return tf.keras.losses.binary_crossentropy(y_true, y_pred)
    def get_config(self):
        cfg = super().get_config(); cfg["smoothing"] = self.smoothing; return cfg


def build_callbacks(model_tag):
    ckpt_p1 = os.path.join(CKPT_DIR, f"{model_tag}_p1.keras")
    ckpt_p2 = os.path.join(CKPT_DIR, f"{model_tag}_p2.keras")
    return (
        ckpt_p1, ckpt_p2,
        [ModelCheckpoint(ckpt_p1, monitor='val_accuracy',
                         save_best_only=True, verbose=0),
         ReduceLROnPlateau(monitor='val_loss', factor=0.4,
                           patience=3, min_lr=1e-7, verbose=0),
         EarlyStopping(monitor='val_accuracy', patience=6,
                       restore_best_weights=True, verbose=1)],
        [ModelCheckpoint(ckpt_p2, monitor='val_accuracy',
                         save_best_only=True, verbose=0),
         LearningRateScheduler(
             lambda ep: float(1e-7 + 0.5*(1e-5-1e-7)*(1+np.cos(
                 np.pi*np.clip((ep-EPOCHS_P1)/max(EPOCHS_P2,1),0,1)))),
             verbose=0),
         ReduceLROnPlateau(monitor='val_loss', factor=0.4,
                           patience=3, min_lr=1e-7, verbose=0),
         EarlyStopping(monitor='val_accuracy', patience=10,
                       restore_best_weights=True, verbose=1)],
    )


def train_two_phase(model, model_tag, custom_objects):
    ckpt_p1, ckpt_p2, cb1, cb2 = build_callbacks(model_tag)

    # ── Phase 1: frozen backbone ──────────────────────────────────────
    model.compile(optimizer=Adam(1e-3), loss=LabelSmoothedBCE(0.05),
                  metrics=[tf.keras.metrics.BinaryAccuracy(name='accuracy'),
                            tf.keras.metrics.AUC(name='auc'),
                            tf.keras.metrics.Precision(name='precision'),
                            tf.keras.metrics.Recall(name='recall')])
    print(f"\n{'='*60}\nPHASE 1 — {model_tag} (backbone frozen)\n{'='*60}")
    h1 = model.fit(train_ds, validation_data=val_ds,
                   epochs=EPOCHS_P1, class_weight=class_weight,
                   callbacks=cb1, verbose=1)
    np.save(os.path.join(HIST_DIR, f"{model_tag}_h1.npy"), h1.history)

    # ── Phase 2: fine-tune top-40 Xception layers ─────────────────────
    model = tf.keras.models.load_model(ckpt_p1, custom_objects=custom_objects)
    backbone = model.get_layer("xception")
    backbone.trainable = True
    freeze_up = len(backbone.layers) - 40
    for l in backbone.layers[:freeze_up]: l.trainable = False
    for l in backbone.layers:
        if isinstance(l, layers.BatchNormalization): l.trainable = False
    model.compile(optimizer=Adam(1e-5), loss=LabelSmoothedBCE(0.05),
                  metrics=[tf.keras.metrics.BinaryAccuracy(name='accuracy'),
                            tf.keras.metrics.AUC(name='auc'),
                            tf.keras.metrics.Precision(name='precision'),
                            tf.keras.metrics.Recall(name='recall')])
    print(f"\n{'='*60}\nPHASE 2 — {model_tag} (top-40 unfrozen)\n{'='*60}")
    h2 = model.fit(train_ds, validation_data=val_ds,
                   epochs=EPOCHS_P1+EPOCHS_P2,
                   initial_epoch=h1.epoch[-1]+1,
                   class_weight=class_weight,
                   callbacks=cb2, verbose=1)
    np.save(os.path.join(HIST_DIR, f"{model_tag}_h2.npy"), h2.history)
    final_path = os.path.join(CKPT_DIR, f"{model_tag}_final.keras")
    model.save(final_path)
    print(f"\u2713 {model_tag} saved \u2192 {final_path}")
    return model, h1, h2

print("\u2713 Training helpers ready")


✓ Training helpers ready


---
## Cell 5 — Evaluation Engine

In [6]:
def evaluate_dataset(model, ds, dataset_name):
    y_true, y_prob = [], []
    for imgs, labels in ds:
        y_prob.extend(model.predict(imgs, verbose=0).flatten())
        y_true.extend(labels.numpy().flatten())
    y_true = np.array(y_true); y_prob = np.array(y_prob)
    y_pred = (y_prob >= 0.5).astype(int)
    cr  = classification_report(y_true, y_pred,
                                  target_names=['Fake','Real'],
                                  output_dict=True)
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    print(f"  {dataset_name:<30}  "
          f"Acc={cr['accuracy']*100:.2f}%  "
          f"AUC={roc_auc_score(y_true,y_prob):.4f}  "
          f"F1={cr['weighted avg']['f1-score']:.4f}")
    return {'name': dataset_name,
            'acc':  cr['accuracy'],
            'auc':  roc_auc_score(y_true, y_prob),
            'prec': cr['weighted avg']['precision'],
            'rec':  cr['weighted avg']['recall'],
            'f1':   cr['weighted avg']['f1-score'],
            'cm':   confusion_matrix(y_true, y_pred),
            'fpr':  fpr, 'tpr': tpr,
            'y_true': y_true, 'y_prob': y_prob, 'y_pred': y_pred}

print("\u2713 Evaluation engine ready")


✓ Evaluation engine ready


---
## Cell 6 — Paper Figure Functions

In [7]:
plt.rcParams.update({
    'font.family':'DejaVu Sans','font.size':11,
    'axes.titlesize':12,'axes.labelsize':11,
    'legend.fontsize':10,'figure.dpi':150,
    'savefig.dpi':300,'savefig.bbox':'tight'})

def save(fig, name):
    fig.savefig(os.path.join(FIG_DIR, f"{name}.pdf"))
    fig.savefig(os.path.join(FIG_DIR, f"{name}.png"))
    plt.show(); plt.close(fig)
    print(f"  \u2713 {name}.pdf/png")


def plot_training(h1_dict, h2_dict, model_tag):
    ep1 = range(1, len(h1_dict['accuracy'])+1)
    ep2 = range(len(h1_dict['accuracy'])+1,
                 len(h1_dict['accuracy'])+len(h2_dict['accuracy'])+1)
    pb  = list(ep1)[-1]+0.5
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    for ax, m, title in zip(axes,['accuracy','auc'],['Accuracy','AUC-ROC']):
        ax.plot(ep1, h1_dict[m],          '#2166ac', lw=1.8, ls='--', label='P1 Train')
        ax.plot(ep1, h1_dict[f'val_{m}'], '#d6604d', lw=1.8, ls='--', label='P1 Val')
        ax.plot(ep2, h2_dict[m],          '#2166ac', lw=2.2,           label='P2 Train')
        ax.plot(ep2, h2_dict[f'val_{m}'], '#d6604d', lw=2.2,           label='P2 Val')
        ax.axvline(pb, color='grey', lw=1.2, ls=':', label='Phase boundary')
        ax.set(xlabel='Epoch', ylabel=title,
               title=f'{title} — {model_tag}', ylim=(0.5,1.01))
        ax.legend(loc='lower right'); ax.grid(alpha=.35)
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    fig.suptitle(f'Training Dynamics — {model_tag}',
                 fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    save(fig, f"fig_{model_tag}_training")


def plot_confusion(results_list, model_tag):
    n = len(results_list)
    fig, axes = plt.subplots(1, n, figsize=(5*n, 4.5))
    if n == 1: axes = [axes]
    cmap = LinearSegmentedColormap.from_list('bl',['#f7fbff','#2171b5'])
    for ax, r in zip(axes, results_list):
        cm  = r['cm']
        pct = cm.astype(float)/cm.sum(axis=1,keepdims=True)*100
        sns.heatmap(cm, annot=False, cmap=cmap, ax=ax, cbar=False,
                    square=True, linewidths=0.5, linecolor='white',
                    xticklabels=['Fake','Real'], yticklabels=['Fake','Real'])
        for i in range(2):
            for j in range(2):
                col = 'white' if pct[i,j]>50 else '#1a1a2e'
                ax.text(j+.5, i+.40, f"{cm[i,j]:,}",
                        ha='center', va='center',
                        fontsize=14, fontweight='bold', color=col)
                ax.text(j+.5, i+.62, f"({pct[i,j]:.1f}%)",
                        ha='center', va='center', fontsize=9, color=col)
        ax.set_title(f"{r['name'].split('(')[0].strip()}\n"
                     f"Acc={r['acc']*100:.2f}%  AUC={r['auc']:.4f}",
                     fontsize=11)
        ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    fig.suptitle(f'Confusion Matrices — {model_tag}',
                 fontsize=13, fontweight='bold', y=1.03)
    plt.tight_layout()
    save(fig, f"fig_{model_tag}_confusion")


def plot_roc(results_list, model_tag):
    cols = ['#2166ac','#d6604d']; ls = ['-','--']
    fig, ax = plt.subplots(figsize=(7,6))
    for r,c,l in zip(results_list, cols, ls):
        ax.plot(r['fpr'], r['tpr'], color=c, lw=2.2, ls=l,
                label=f"{r['name'].split('(')[0].strip()} (AUC={r['auc']:.4f})")
    ax.plot([0,1],[0,1],'k--',lw=1.2,alpha=.6,label='Random (0.5000)')
    ax.set(xlim=(-0.01,1.01), ylim=(-0.01,1.02),
           xlabel='False Positive Rate', ylabel='True Positive Rate',
           title=f'ROC Curves — {model_tag}')
    ax.legend(loc='lower right'); ax.grid(alpha=.3)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    plt.tight_layout()
    save(fig, f"fig_{model_tag}_roc")


def plot_pr(results_list, model_tag):
    cols = ['#2166ac','#d6604d']; ls = ['-','--']
    fig, ax = plt.subplots(figsize=(7,6))
    for r,c,l in zip(results_list, cols, ls):
        p,rec,_ = precision_recall_curve(r['y_true'], r['y_prob'])
        ap = average_precision_score(r['y_true'], r['y_prob'])
        ax.plot(rec, p, color=c, lw=2.2, ls=l,
                label=f"{r['name'].split('(')[0].strip()} (AP={ap:.4f})")
    ax.set(xlim=(-0.01,1.01), ylim=(-0.01,1.05),
           xlabel='Recall', ylabel='Precision',
           title=f'Precision-Recall Curves — {model_tag}')
    ax.legend(loc='lower left'); ax.grid(alpha=.3)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    plt.tight_layout()
    save(fig, f"fig_{model_tag}_pr")


def plot_score_dist(results_list, model_tag):
    n = len(results_list)
    fig, axes = plt.subplots(1, n, figsize=(5*n, 4))
    if n == 1: axes = [axes]
    for ax, r in zip(axes, results_list):
        ax.hist(r['y_prob'][r['y_true']==0], bins=40, color='#d6604d',
                alpha=.70, density=True, label='Fake', edgecolor='white', lw=.4)
        ax.hist(r['y_prob'][r['y_true']==1], bins=40, color='#2166ac',
                alpha=.70, density=True, label='Real', edgecolor='white', lw=.4)
        ax.axvline(.5, color='black', lw=1.5, ls='--', label='Threshold')
        ax.set(xlabel='P(fake)', ylabel='Density',
               title=r['name'].split('(')[0].strip())
        ax.legend(fontsize=9); ax.grid(alpha=.3)
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    fig.suptitle(f'Score Distributions — {model_tag}',
                 fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    save(fig, f"fig_{model_tag}_scores")


def plot_metric_bar(results_list, model_tag):
    metrics = ['Accuracy','AUC','Precision','Recall','F1']
    keys    = ['acc','auc','prec','rec','f1']
    cols    = ['#2166ac','#d6604d']
    x = np.arange(len(metrics)); w = 0.30
    fig, ax = plt.subplots(figsize=(10, 5))
    for i,(r,c) in enumerate(zip(results_list, cols)):
        vals = [r[k] for k in keys]
        bars = ax.bar(x+i*w, vals, w, label=r['name'].split('(')[0].strip(),
                      color=c, alpha=.85, edgecolor='white', lw=.6)
        for b,v in zip(bars,vals):
            ax.text(b.get_x()+b.get_width()/2, b.get_height()+.005,
                    f"{v:.3f}", ha='center', va='bottom',
                    fontsize=7.5, rotation=90)
    ax.set_xticks(x+w/2); ax.set_xticklabels(metrics, fontsize=11)
    ax.set(ylabel='Score', ylim=(0,1.14),
           title=f'Performance Comparison — {model_tag}')
    ax.legend(loc='lower right'); ax.grid(axis='y',alpha=.35)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    plt.tight_layout()
    save(fig, f"fig_{model_tag}_metrics")

print("\u2713 All figure functions ready")


✓ All figure functions ready


---
## Cell 7 — Grad-CAM Engine

> Targets `block14_sepconv2_act` inside the nested Xception sub-model.
> Uses `tf.identity()` to avoid the `Variable /= value` crash in XceptionPreprocess.

In [8]:
GRADCAM_LAYER = "block14_sepconv2_act"


def get_grad_model(model):
    """Build grad_model by drilling into the nested Xception sub-model."""
    xception_sub = model.get_layer("xception")
    return (
        xception_sub,
        tf.keras.Model(
            inputs  = xception_sub.input,
            outputs = [xception_sub.get_layer(GRADCAM_LAYER).output,
                       xception_sub.output]
        )
    )


def compute_gradcam(full_model, grad_model, img_array):
    """
    img_array : numpy (1, 299, 299, 3) uint8, RGB order
    Returns: cam_n, sal_n, fft_mag, pred_score
    """
    img_tensor = tf.cast(img_array, tf.float32)

    # ── Full-model prediction score ───────────────────────────────────
    pred_score = float(full_model(img_tensor, training=False).numpy()[0, 0])

    # ── Spatial Grad-CAM via Xception sub-model ───────────────────────
    x_pre = tf.keras.applications.xception.preprocess_input(
        tf.cast(img_array, tf.float32))
    with tf.GradientTape() as tape:
        conv_out, xception_out = grad_model(x_pre, training=False)
        loss = tf.reduce_mean(xception_out)
    grads  = tape.gradient(loss, conv_out)
    pooled = tf.reduce_mean(grads, axis=[1, 2])
    cam    = tf.reduce_sum(conv_out * pooled[:, tf.newaxis, tf.newaxis, :], axis=-1)
    cam    = tf.squeeze(tf.nn.relu(cam), 0).numpy()
    cam_r  = cv2.resize(cam, (299, 299))
    cam_n  = (cam_r - cam_r.min()) / (cam_r.max() - cam_r.min() + 1e-8)

    # ── Input-gradient saliency (tf.identity avoids Variable /= crash) ──
    img_v = tf.Variable(img_tensor)
    with tf.GradientTape() as tape2:
        tape2.watch(img_v)
        pred2 = full_model(tf.identity(img_v), training=False)
        loss2 = pred2[:, 0]
    img_grads = tape2.gradient(loss2, img_v)
    sal   = tf.squeeze(tf.reduce_max(tf.abs(img_grads), axis=-1), 0).numpy()
    sal_n = (sal - sal.min()) / (sal.max() - sal.min() + 1e-8)

    # ── FFT magnitude (display only) ─────────────────────────────────
    gray = cv2.cvtColor(img_array[0], cv2.COLOR_RGB2GRAY).astype(np.float32) / 255.0
    fft  = np.fft.fftshift(np.fft.fft2(gray))
    mag  = np.log1p(np.abs(fft))
    mag  = (mag - mag.min()) / (mag.max() - mag.min() + 1e-8)

    return cam_n, sal_n, mag, pred_score


def overlay(img_rgb, cam, alpha=0.45):
    hm      = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
    hm_rgb  = cv2.cvtColor(hm, cv2.COLOR_BGR2RGB)
    return cv2.addWeighted(img_rgb, 1 - alpha, hm_rgb, alpha, 0)

print("\u2713 Grad-CAM engine ready")


✓ Grad-CAM engine ready


---
## Cell 8 — Grad-CAM Figure Generator

In [9]:
def gradcam_figure(full_model, ds, results_dict,
                    dataset_name, model_tag, n=5):
    """
    Generate Grad-CAM figures for n correctly-classified real and fake samples.
    Layout: 4 rows (Original | Grad-CAM | FFT | Saliency) x n cols.
    """
    _, grad_model = get_grad_model(full_model)
    collected = {'real': [], 'fake': []}

    for imgs, labels in ds:
        batch_probs = full_model.predict(imgs, verbose=0).flatten()
        for i in range(len(imgs)):
            img_np   = imgs[i].numpy().astype(np.uint8)  # RGB (loaded by tf.keras)
            lbl      = int(labels[i].numpy())
            cls      = 'real' if lbl == 1 else 'fake'
            pred_lbl = int(batch_probs[i] >= 0.5)
            if pred_lbl == lbl and len(collected[cls]) < n:
                collected[cls].append(img_np)
        if len(collected['real']) >= n and len(collected['fake']) >= n:
            break

    row_titles = ['Original', 'Grad-CAM overlay', 'FFT spectrum', 'Saliency map']
    colours    = {'real': '#2166ac', 'fake': '#d6604d'}

    for cls, samples in collected.items():
        if not samples:
            print(f"  \u26a0 no correctly-classified {cls} samples — skipping")
            continue

        n_cols = min(len(samples), n)
        fig = plt.figure(figsize=(n_cols * 2.8, 4 * 2.8))
        gs  = gridspec.GridSpec(4, n_cols, hspace=0.05, wspace=0.05)

        for ci, img_rgb in enumerate(samples[:n_cols]):
            cam, sal, fft_map, score = compute_gradcam(
                full_model, grad_model, img_rgb[np.newaxis, ...])
            visuals = [img_rgb, overlay(img_rgb, cam), fft_map, sal]

            for ri, vis in enumerate(visuals):
                ax = fig.add_subplot(gs[ri, ci])
                ax.imshow(vis, cmap=('magma'   if ri == 3 else
                                      'inferno' if ri == 2 else None))
                ax.axis('off')
                if ci == 0:
                    ax.set_ylabel(row_titles[ri], fontsize=8,
                                   rotation=90, labelpad=4, va='center')
                    ax.yaxis.set_label_position('left')
                if ri == 0:
                    ax.set_title(f"P(fake)={score:.3f}", fontsize=8,
                                  pad=3, color=colours[cls])

        ds_short = dataset_name.replace(' ','_').replace('+','plus')
        fig.suptitle(
            f"{dataset_name} — {cls.upper()} samples\n"
            f"Grad-CAM \u00b7 FFT \u00b7 Saliency  [{model_tag}]",
            fontsize=11, fontweight='bold', color=colours[cls], y=1.01)
        plt.tight_layout()
        save(fig, f"fig_{model_tag}_gradcam_{ds_short}_{cls}")

print("\u2713 Grad-CAM figure generator ready")


✓ Grad-CAM figure generator ready


---
## Cell 9 — Custom Layers: FFT

In [10]:
class XceptionPreprocess(layers.Layer):
    def call(self, x):
        return tf.keras.applications.xception.preprocess_input(x)
    def get_config(self): return super().get_config()

class FrequencyStream(layers.Layer):
    def __init__(self, embed_dim=256, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.conv1= layers.Conv2D(64,  3, activation='relu', padding='same')
        self.bn1  = layers.BatchNormalization()
        self.conv2= layers.Conv2D(128, 3, strides=2, activation='relu', padding='same')
        self.bn2  = layers.BatchNormalization()
        self.conv3= layers.Conv2D(embed_dim, 3, strides=2, activation='relu', padding='same')
        self.bn3  = layers.BatchNormalization()
        self.gap  = layers.GlobalAveragePooling2D()
    def call(self, x, training=False):
        gray    = tf.squeeze(tf.image.rgb_to_grayscale(x/255.), axis=-1)
        fft     = tf.signal.fftshift(
                    tf.signal.fft2d(tf.cast(gray, tf.complex64)))
        mag     = tf.math.log1p(tf.abs(fft))
        mn      = tf.reduce_min(mag, axis=[1,2], keepdims=True)
        mx      = tf.reduce_max(mag, axis=[1,2], keepdims=True)
        mag     = (mag-mn)/(mx-mn+1e-8)
        fft_img = tf.stack([mag,mag,mag], axis=-1)
        h = self.bn1(self.conv1(fft_img), training=training)
        h = self.bn2(self.conv2(h),       training=training)
        h = self.bn3(self.conv3(h),       training=training)
        return self.gap(h)
    def get_config(self):
        cfg=super().get_config(); cfg["embed_dim"]=self.embed_dim; return cfg

CUSTOM_OBJECTS_3 = {
    "XceptionPreprocess": XceptionPreprocess,
    "FrequencyStream":    FrequencyStream,
    "LabelSmoothedBCE":   LabelSmoothedBCE,
}
print("\u2713 Xception+FFT custom layers defined")


✓ Xception+FFT custom layers defined


---
## Cell 10 — Build Model: Xception + FFT

In [11]:
def build_xception_fft(img_size=299):
    inp      = layers.Input(shape=(img_size, img_size, 3), name="input")

    # ── Stream A: Xception spatial ────────────────────────────────────
    x_pre    = XceptionPreprocess(name="xception_preprocess")(inp)
    backbone = Xception(weights='imagenet', include_top=False,
                        input_shape=(img_size, img_size, 3))
    backbone.trainable = False
    x_rgb = backbone(x_pre, training=False)
    x_rgb = layers.GlobalAveragePooling2D(name="xception_gap")(x_rgb)
    x_rgb = layers.Dense(256, activation='relu',
                          kernel_regularizer=l2(1e-4),
                          name="xception_proj")(x_rgb)
    x_rgb = layers.Dropout(0.3, name="drop_rgb")(x_rgb)

    # ── Stream B: FFT frequency ───────────────────────────────────────
    x_fft = FrequencyStream(embed_dim=256, name="fft_stream")(inp)
    x_fft = layers.Dense(256, activation='relu',
                          kernel_regularizer=l2(1e-4),
                          name="fft_proj")(x_fft)
    x_fft = layers.Dropout(0.3, name="drop_fft")(x_fft)

    # ── Concatenate and classify ──────────────────────────────────────
    fused = layers.Concatenate(name="fusion")([x_rgb, x_fft])
    out   = layers.Dense(256, activation='relu',
                          kernel_regularizer=l2(1e-4),
                          name="head_dense1")(fused)
    out   = layers.Dropout(0.4, name="drop_head1")(out)
    out   = layers.Dense(64,  activation='relu', name="head_dense2")(out)
    out   = layers.Dropout(0.3, name="drop_head2")(out)
    output= layers.Dense(1, activation='sigmoid', name="output")(out)

    return Model(inputs=inp, outputs=output, name="Xception_FFT")

model_3 = build_xception_fft()
model_3.summary()
print(f"Trainable params: "
      f"{sum(np.prod(v.shape) for v in model_3.trainable_variables):,}")


83683744/83683744 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "Xception_FFT"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input (InputLayer)  │ (None, 299, 299,  │          0 │ -                 │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ xception_preprocess │ (None, 299, 299,  │          0 │ input[0][0]       │
│ (XceptionPreproces… │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ xception            │ (None, 10, 10,    │ 20,861,480 │ xception_preproc… │
│ (Functional)        │ 2048)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ xception_gap        │ (None, 2048)      │          0 │ xception[0][0]    │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fft_stream          │ (None, 256)       │    372,608 │ input[0][0]       │
│ (FrequencyStream)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ xception_proj       │ (None, 256)       │    524,544 │ xception_gap[0][… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fft_proj (Dense)    │ (None, 256)       │     65,792 │ fft_stream[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ drop_rgb (Dropout)  │ (None, 256)       │          0 │ xception_proj[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ drop_fft (Dropout)  │ (None, 256)       │          0 │ fft_proj[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fusion              │ (None, 512)       │          0 │ drop_rgb[0][0],   │
│ (Concatenate)       │                   │            │ drop_fft[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ head_dense1 (Dense) │ (None, 256)       │    131,328 │ fusion[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ drop_head1          │ (None, 256)       │          0 │ head_dense1[0][0] │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ head_dense2 (Dense) │ (None, 64)        │     16,448 │ drop_head1[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ drop_head2          │ (None, 64)        │          0 │ head_dense2[0][0] │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output (Dense)      │ (None, 1)         │         65 │ drop_head2[0][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 21,972,265 (83.82 MB)

 Trainable params: 1,109,889 (4.23 MB)

 Non-trainable params: 20,862,376 (79.58 MB)

Trainable params: 1,109,889


---
## Cell 11 — Train

In [12]:
model_3, h3_p1, h3_p2 = train_two_phase(
    model_3, "xception_fft", CUSTOM_OBJECTS_3)
print("\u2713 Xception+FFT training complete")



PHASE 1 — xception_fft (backbone frozen)
Epoch 1/15


I0000 00:00:1776773478.181302      75 service.cc:152] XLA service 0x79a614415800 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1776773478.181359      75 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1776773478.181366      75 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1776773479.899380      75 cuda_dnn.cc:529] Loaded cuDNN version 91002
2026-04-21 12:11:26.843945: E external/local_xla/xla/service/slow_operation_alarm.cc:73] Trying algorithm eng3{k11=2} for conv %cudnn-conv.84 = (f32[32,128,147,147]{3,2,1,0}, u8[0]{0}) custom-call(f32[32,128,147,147]{3,2,1,0} %bitcast.18013, f32[128,1,3,3]{3,2,1,0} %bitcast.18017), window={size=3x3 pad=1_1x1_1}, dim_labels=bf01_oi01->bf01, feature_group_count=128, custom_call_target="__cudnn$convForward", metadata={op_type="DepthwiseConv2dNative" op_name="Xception_FFT_1/xception_1/block2_sepconv2_1/separable_con

298/438 ━━━━━━━━━━━━━━━━━━━━ 1:21 584ms/step - accuracy: 0.5005 - auc: 0.5019 - loss: 0.8045 - precision: 0.4879 - recall: 0.4415

2026-04-21 12:15:12.868214: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-21 12:15:13.091373: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-21 12:15:15.743586: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-21 12:15:15.935252: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-21 12:15:17.616913: E external/local_xla/xla/stream_

438/438 ━━━━━━━━━━━━━━━━━━━━ 0s 676ms/step - accuracy: 0.5045 - auc: 0.5063 - loss: 0.7963 - precision: 0.4951 - recall: 0.4595

...


PHASE 2 — xception_fft (top-40 unfrozen)
Epoch 16/45


2026-04-21 13:54:41.903991: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-21 13:54:42.058164: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-21 13:54:43.570025: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-21 13:54:43.710599: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-21 13:54:44.481467: E external/local_xla/xla/stream_

 37/438 ━━━━━━━━━━━━━━━━━━━━ 4:45 712ms/step - accuracy: 0.6945 - auc: 0.7648 - loss: 0.5967 - precision: 0.6957 - recall: 0.7173

2026-04-21 13:55:32.257794: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-21 13:55:32.400618: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-21 13:55:32.960528: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-21 13:55:33.102298: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-21 13:55:33.240102: E external/local_xla/xla/stream_

438/438 ━━━━━━━━━━━━━━━━━━━━ 536s 878ms/step - accuracy: 0.7179 - auc: 0.8033 - loss: 0.5653 - precision: 0.7158 - recall: 0.7280 - val_accuracy: 0.7653 - val_auc: 0.8621 - val_loss: 0.5041 - val_precision: 0.7795 - val_recall: 0.7400 - learning_rate: 1.0000e-05
Epoch 17/45
438/438 ━━━━━━━━━━━━━━━━━━━━ 455s 769ms/step - accuracy: 0.8106 - auc: 0.8993 - loss: 0.4574 - precision: 0.8072 - recall: 0.8164 - val_accuracy: 0.8117 - val_auc: 0.9062 - val_loss: 0.4537 - val_precision: 0.7863 - val_recall: 0.8560 - learning_rate: 9.9729e-06
Epoch 18/45
438/438 ━━━━━━━━━━━━━━━━━━━━ 455s 769ms/step - accuracy: 0.8646 - auc: 0.9425 - loss: 0.3811 - precision: 0.8653 - recall: 0.8624 - val_accuracy: 0.8627 - val_auc: 0.9462 - val_loss: 0.3715 - val_precision: 0.8826 - val_recall: 0.8367 - learning_rate: 9.8918e-06
Epoch 19/45
438/438 ━━━━━━━━━━━━━━━━━━━━ 455s 768ms/step - accuracy: 0.8922 - auc: 0.9611 - loss: 0.3397 - precision: 0.8943 - recall: 0.8901 - val_accuracy: 0.8757 - val_auc: 0.9564 - va

---
## Cell 12 — Evaluate + Generate All Figures

In [13]:
print("\nEvaluating Xception+FFT...")
r3_ff    = evaluate_dataset(model_3, fftest_ds,  "FF++ C23 (in-distribution)")
r3_celeb = evaluate_dataset(model_3, celeb_ds,   "Celeb-DF v2 (zero-shot)")
all_results = [r3_ff, r3_celeb]

h1d = np.load(os.path.join(HIST_DIR,"xception_fft_h1.npy"), allow_pickle=True).item()
h2d = np.load(os.path.join(HIST_DIR,"xception_fft_h2.npy"), allow_pickle=True).item()

plot_training(h1d, h2d, "xception_fft")
plot_confusion(all_results, "xception_fft")
plot_roc(all_results, "xception_fft")
plot_pr(all_results, "xception_fft")
plot_score_dist(all_results, "xception_fft")
plot_metric_bar(all_results, "xception_fft")
gradcam_figure(model_3, fftest_ds,  r3_ff,    "FF++ C23",    "xception_fft", NUM_GRADCAM)
gradcam_figure(model_3, celeb_ds,   r3_celeb, "Celeb-DF v2", "xception_fft", NUM_GRADCAM)
print("\u2713 All figures saved")



Evaluating Xception+FFT...
  FF++ C23 (in-distribution)      Acc=89.63%  AUC=0.9674  F1=0.8961
  Celeb-DF v2 (zero-shot)         Acc=63.87%  AUC=0.7220  F1=0.6187
  ✓ fig_xception_fft_training.pdf/png
  ✓ fig_xception_fft_confusion.pdf/png
  ✓ fig_xception_fft_roc.pdf/png
  ✓ fig_xception_fft_pr.pdf/png
  ✓ fig_xception_fft_scores.pdf/png
  ✓ fig_xception_fft_metrics.pdf/png


/tmp/ipykernel_23/4232300129.py:14: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  lbl      = int(labels[i].numpy())


/tmp/ipykernel_23/4232300129.py:57: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


  ✓ fig_xception_fft_gradcam_FFplusplus_C23_real.pdf/png


/tmp/ipykernel_23/4232300129.py:57: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


...

/tmp/ipykernel_23/4232300129.py:14: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  lbl      = int(labels[i].numpy())


/tmp/ipykernel_23/4232300129.py:57: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


  ✓ fig_xception_fft_gradcam_Celeb-DF_v2_real.pdf/png


/tmp/ipykernel_23/4232300129.py:57: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


  ✓ fig_xception_fft_gradcam_Celeb-DF_v2_fake.pdf/png
✓ All figures saved


---
## Cell 13 — Summary

In [14]:
print("\n" + "="*72)
print("  PAPER FIGURES — xception_fft")
print("="*72)
for f in sorted(os.listdir(FIG_DIR)):
    sz = os.path.getsize(os.path.join(FIG_DIR,f))/1024
    print(f"  {f:<52} {sz:>6.1f} KB")
print("="*72)

print("\n  RESULTS TABLE — xception_fft")
print("="*72)
print(f"  {'Dataset':<30} {'Acc%':>7} {'AUC':>7} {'Prec':>7} {'Rec':>7} {'F1':>7}")
print("-"*72)
for r in all_results:
    tag = "(in-dist.)" if "FF++" in r['name'] else "(zero-shot)"
    print(f"  {r['name'].split('(')[0].strip():<30} "
          f"{r['acc']*100:>7.2f} {r['auc']:>7.4f} "
          f"{r['prec']:>7.4f} {r['rec']:>7.4f} {r['f1']:>7.4f}  {tag}")
print("="*72)
print("\n\u2713 Done. Download /kaggle/working/paper_figures/")



  PAPER FIGURES — xception_fft
  fig_xception_fft_confusion.pdf                         24.4 KB
  fig_xception_fft_confusion.png                        143.0 KB
  fig_xception_fft_gradcam_Celeb-DF_v2_fake.pdf        6797.3 KB
  fig_xception_fft_gradcam_Celeb-DF_v2_fake.png        7665.4 KB
  fig_xception_fft_gradcam_Celeb-DF_v2_real.pdf        7256.7 KB
  fig_xception_fft_gradcam_Celeb-DF_v2_real.png        8218.2 KB
  fig_xception_fft_gradcam_FFplusplus_C23_fake.pdf     7584.4 KB
  fig_xception_fft_gradcam_FFplusplus_C23_fake.png     8569.3 KB
  fig_xception_fft_gradcam_FFplusplus_C23_real.pdf     7358.3 KB
  fig_xception_fft_gradcam_FFplusplus_C23_real.png     8319.9 KB
  fig_xception_fft_metrics.pdf                           16.3 KB
  fig_xception_fft_metrics.png                          105.2 KB
  fig_xception_fft_pr.pdf                                35.6 KB
  fig_xception_fft_pr.png                               140.5 KB
  fig_xception_fft_roc.pdf                               3